# Colab: Experiments 1b and 2 (model-selectable)

This notebook runs the current experiment pipelines on Colab with either `meta-llama/Llama-3.2-3B` (base) or `gpt2-large`.

Important notes:
- Supported choices are `llama-3.2-3B-base`, `llama-3.2-3B`, and `gpt2-large`.
- If you choose Llama, you must accept the Meta Llama license on Hugging Face and provide a Colab secret named `HF_TOKEN`.
- If you choose GPT-2, Hugging Face login is skipped.
- Optional core lexical-overlap comparison is supported for `1b` and `2`.
- Jabberwocky is still included inside those runs, but there is no separate Jabberwocky lexical-overlap manipulation here.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/Geometry-of-Syntax')
REPO_URL = 'https://github.com/<YOUR-USERNAME>/Geometry-of-Syntax.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Geometry-of-Syntax
!git pull --ff-only || true
!nvidia-smi || true
!python --version

# Keep Colab's CUDA torch. Install only Python-side dependencies used by the repo.
!pip install -q pandas scipy statsmodels tqdm accelerate sentencepiece huggingface_hub "transformers>=4.57,<5"


In [ ]:
from huggingface_hub import login

# Choose one: 'llama-3.2-3B-base', 'llama-3.2-3B', or 'gpt2-large'
MODEL_CHOICE = 'llama-3.2-3B-base'

MODEL_OPTIONS = {
    'llama-3.2-3B-base': 'meta-llama/Llama-3.2-3B',
    'llama-3.2-3B': 'meta-llama/Llama-3.2-3B',
    'gpt2-large': 'gpt2-large',
}
if MODEL_CHOICE not in MODEL_OPTIONS:
    raise ValueError(f'Unknown MODEL_CHOICE: {MODEL_CHOICE}. Choose one of: {sorted(MODEL_OPTIONS)}')

MODEL_NAME = MODEL_OPTIONS[MODEL_CHOICE]

if MODEL_NAME == 'meta-llama/Llama-3.2-3B':
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None

    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login loaded from Colab secret HF_TOKEN.')
    else:
        print('No HF_TOKEN secret found. Add one in Colab after accepting the Meta Llama license, then rerun this cell.')
else:
    print(f'Selected {MODEL_CHOICE}; skipping Hugging Face login.')

print({'model_choice': MODEL_CHOICE, 'model_name': MODEL_NAME})


In [ ]:
DEVICE = 'cuda'
TORCH_DTYPE = 'float16'
SEED = 13
MAX_ITEMS = 2080

RUN_EXPERIMENT_1B = True
RUN_EXPERIMENT_2 = True


# Turn this on only if you explicitly want the contaminated core comparison too.
RUN_CORE_LEXICAL_OVERLAP_COMPARISON = False

# Conservative defaults for Colab GPU memory.
BATCH_SIZE_1B = 4
BATCH_SIZE_2 = 4


MODEL_TAG = MODEL_CHOICE.replace('.', '').replace('-', '_')
BASE_OUTPUT_ROOT = f'behavioral_results/colab_{MODEL_TAG}'
CORE_PRIME_MODES = ['lexically_controlled']
if RUN_CORE_LEXICAL_OVERLAP_COMPARISON:
    CORE_PRIME_MODES.append('lexical_overlap')

EXP1B_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-1/experiment-1b/processing_experiment_1b_{MODEL_TAG}_v1_{mode}' for mode in CORE_PRIME_MODES
}
EXP2_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-2/experiment-2_{mode}' for mode in CORE_PRIME_MODES
}

EXP1B_PRIME_CONDITIONS = ['active', 'passive', 'no_prime_eos', 'no_prime_empty', 'filler']

# Best current Experiment 2 wording from the local prompt screen.
EXP2_EVENT_STYLE = 'involving_event'
EXP2_ROLE_STYLE = 'did_to'
EXP2_QUOTE_STYLE = 'mary_answered'
EXP2_PRIME_CONDITIONS = ['active', 'passive', 'no_demo', 'filler']

print({
    'model_choice': MODEL_CHOICE,
    'model_name': MODEL_NAME,
    'device': DEVICE,
    'torch_dtype': TORCH_DTYPE,
    'max_items': MAX_ITEMS,
    'core_prime_modes': CORE_PRIME_MODES,
    'run_1b': RUN_EXPERIMENT_1B,
    'run_2': RUN_EXPERIMENT_2,
    'exp1b_output_roots': EXP1B_OUTPUT_ROOTS,
    'exp2_output_roots': EXP2_OUTPUT_ROOTS,
})


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_1B:
    for core_mode in CORE_PRIME_MODES:
        which_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/16_run_processing_experiment_1b.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_1B),
            '--max-items', str(MAX_ITEMS),
            '--which', which_for_mode,
            '--core-prime-mode', core_mode,
            '--seed', str(SEED),
            '--output-root', EXP1B_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP1B_PRIME_CONDITIONS,
        ])
else:
    print('Skipping Experiment 1b.')


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_2:
    for core_mode in CORE_PRIME_MODES:
        which_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/25_run_demo_prompt_experiments.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_2),
            '--max-items', str(MAX_ITEMS),
            '--which', which_for_mode,
            '--core-prime-mode', core_mode,
            '--quote-style', EXP2_QUOTE_STYLE,
            '--event-style', EXP2_EVENT_STYLE,
            '--role-style', EXP2_ROLE_STYLE,
            '--seed', str(SEED),
            '--output-root', EXP2_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP2_PRIME_CONDITIONS,
        ])
else:
    print('Skipping Experiment 2.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

roots = []
if RUN_EXPERIMENT_1B:
    roots.extend(Path(REPO_DIR) / root for root in EXP1B_OUTPUT_ROOTS.values())
if RUN_EXPERIMENT_2:
    roots.extend(Path(REPO_DIR) / root for root in EXP2_OUTPUT_ROOTS.values())

for root in roots:
    if root.exists():
        subprocess.run([sys.executable, 'scripts/20_report_choice_runs.py', '--root', str(root)], check=True)
        print(f'Reported: {root}')
    else:
        print(f'Skipping missing root: {root}')


In [ ]:
from pathlib import Path
import pandas as pd

groups = []
if RUN_EXPERIMENT_1B:
    groups.extend((f'Experiment 1b [{mode}]', Path(REPO_DIR) / root) for mode, root in EXP1B_OUTPUT_ROOTS.items())
if RUN_EXPERIMENT_2:
    groups.extend((f'Experiment 2 [{mode}]', Path(REPO_DIR) / root) for mode, root in EXP2_OUTPUT_ROOTS.items())

for label, root in groups:
    print(f'\n=== {label}: {root} ===')
    priming_path = root / 'priming_framed_results.csv'
    if priming_path.exists():
        display(pd.read_csv(priming_path))
    else:
        print('No priming_framed_results.csv found yet.')


In [ ]:
from pathlib import Path
import shutil

bundle_root = Path(REPO_DIR) / BASE_OUTPUT_ROOT
archive_path = shutil.make_archive('/content/colab_llama32_results', 'zip', bundle_root)
print('Created:', archive_path)
